<div style="background-color: #ADD8E6; border: 1px solid gray; padding: 3px">
    <h3>GraphRAG Index Generation</h3>
    The following is an overview of the workflow:
    <ul>
    <li>Uses Microsoft GraphRAG library to construct a GraphRAG index for the synthetically generated dataset:</li>
        <ul>
            <li>Uses reformatted code-to-summary pairs as input</li>
            <li>Uses openai/gpt-oss-20b at the chat model</li>
            <li>Uses intfloat/e5-mistral-7b-instruct as the embedding model</li>
        </ul>
    </li>
    <li>Stores index in LanceDB database backed by Minio bucket</li>
    </ul>
</div>

In [1]:
##############################################
# Imports
##############################################
from minio import Minio
import os
import lancedb
from datasets import load_dataset
import traceback
import subprocess
import tracemalloc
tracemalloc.start()
import nest_asyncio
nest_asyncio.apply()
import utils

In [2]:
##############################################
# Instance Variables
##############################################
graph_rag_source_path = "graph_rag/source"

graph_rag_jsonfiles_source_path = "graph_rag/source/input"

graph_rag_config_path = "graph_rag/source/settings.yaml"

graph_rag_target_path = "graph_rag/target"

graph_rag_target_path_jsonl = "graph_rag/json"

bucket_name = "data"

In [3]:
##############################################
# Generate GraphRAG index and store in LanceDB
##############################################

try:
    os.makedirs(graph_rag_source_path, exist_ok=True)
    
    os.makedirs(graph_rag_jsonfiles_source_path, exist_ok=True)
    
    os.makedirs(graph_rag_target_path, exist_ok=True)
    
    os.makedirs(graph_rag_target_path_jsonl, exist_ok=True)

    updated_dataset = utils.postprocess_dataset("oaawofolu/emerson", graph_rag_target_path_jsonl)

    utils.split_jsonl_into_json_files(f"{graph_rag_target_path_jsonl}/graphrag.jsonl", graph_rag_jsonfiles_source_path)
    
    db = lancedb.connect(f"s3://{bucket_name}/lancedb-graphrag3",
                         
        storage_options={
            "endpoint_url": os.getenv("AWS_S3_ENDPOINT"),
            
            "aws_access_key_id": os.getenv("AWS_SECRET_ACCESS_KEY"),
            
            "aws_secret_access_key": os.getenv("AWS_ACCESS_KEY_ID"),
            
            "s3_force_path_style": "true",
            
            "allow_http": "true",
        }
    )

    result = subprocess.run(["bash", "graphrag.sh", graph_rag_source_path, graph_rag_config_path], capture_output=True, text=True, check=False)
        
    print(f"\nSubprocess output: {result.stdout}")
    
    if result.stderr:
        
        raise Exception(f"Error processing GraphRag command: {result.stderr}")
    
except Exception as e:
    
    print(f"Error processing GraphRAG DB: {e.stderr}")
    traceback.print_exc()

Generating train split:   0%|          | 0/1796 [00:00<?, ? examples/s]

Map:   0%|          | 0/1796 [00:00<?, ? examples/s]

Map:   0%|          | 0/1796 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]


Subprocess output: Copying settings.yaml...
Initializing GraphRAG index...
Configuring prompts...
/opt/app-root/lib64/python3.11/site-packages/litellm/llms/custom_httpx/async_client_cleanup.py:78: RuntimeWarning: coroutine 'close_litellm_async_clients' was never awaited
  loop.close()
Populating GraphRAG index...
Starting pipeline with workflows: load_input_documents, create_base_text_units, create_final_documents, extract_graph_nlp, prune_graph, finalize_graph, create_communities, create_final_text_units, create_community_reports_text, generate_text_embeddings
Starting workflow: load_input_documents

Workflow complete: load_input_documents
Starting workflow: create_base_text_units
  1 / 1 ............................................................................................
  1 / 1 ............................................................................................
  1 / 1 ............................................................................................
  1 /

In [4]:
##############################################
# Upload data to LanceDB
##############################################
from minio import Minio
import os
import lancedb
from datasets import load_dataset
import nest_asyncio
import pandas as pd
nest_asyncio.apply()


try:
    
    graph_rag_source_path = "graph_rag/source/output"
    
    graph_rag_jsonfiles_source_path = "graph_rag/source/input"
    
    graph_rag_config_path = "graph_rag/source/settings.yaml"
    
    graph_rag_target_path = "graph_rag/target"
    
    bucket_name = "data"
    
    db = lancedb.connect(f"s3://{bucket_name}/lancedb-graphrag3",
                         
        storage_options={
            "endpoint_url": os.getenv("AWS_S3_ENDPOINT"),
            
            "aws_access_key_id": os.getenv("AWS_ACCESS_KEY_ID"),
            
            "aws_secret_access_key": os.getenv("AWS_SECRET_ACCESS_KEY"),
            
            "s3_force_path_style": "true",
            
            "allow_http": "true",
        }
    )

    local_db = lancedb.connect(f"{graph_rag_source_path}/lancedb")

    all_tables = local_db.table_names()

    # Migrate Global Search tables
    print("Migrating global search tables...")
    
    for table_name in all_tables:

        try:

            local_table = local_db.open_table(table_name)
    
            local_df = local_table.to_pandas()
    
            db.create_table(table_name, data=local_df)
    
            print(f"{local_table} migrated.")
    
        except Exception as e:
            
            print(f"Error processing GraphRAG migration to Minio: {e}") 

    # Migrate Local Search tables
    print("Migrating local search tables...")
    
    for file_path in os.listdir(f"{graph_rag_source_path}"):
        
        if file_path.endswith(".parquet"):

            try:
        
                full_path = os.path.join(graph_rag_source_path, file_path)
    
                local_df = pd.read_parquet(full_path)

                print(file_path)

                table_name = file_path.split(".", 1)[0]
    
                db.create_table(table_name, data=local_df)
    
                print(f"{table_name} migrated.")
    
            except Exception as e:
                
                print(f"Error processing GraphRAG migration to Minio: {e}")    
        
    print("Migration complete.")
    
except Exception as e:
    
    print(f"Error processing GraphRAG migration to Minio: {e}")

Migrating global search tables...
Migrating local search tables...
text_units.parquet
text_units migrated.
relationships.parquet
relationships migrated.
entities.parquet
entities migrated.
documents.parquet
documents migrated.
communities.parquet
communities migrated.
Migration complete.


In [5]:
##############################################
# Test Query
##############################################
# import os
# import pandas as pd
# import tiktoken
# import asyncio
# from graphrag.query.indexer_adapters import read_indexer_entities, read_indexer_reports
# from graphrag.query.llm.oai.chat_openai import ChatOpenAI
# from graphrag.query.llm.oai.typing import OpenaiApiType
# from graphrag.query.structured_search.global_search.community_context import (
#     GlobalCommunityContext,
# )
# from graphrag.query.structured_search.global_search.search import GlobalSearch

# # ## Global Search example
# api_key = os.environ["GRAPHRAG_API_KEY"] = "apikey"
# llm_model = "gpt-3.5-turbo"

# llm = ChatOpenAI(
#     api_key=os.getenv("OPENROUTER_TOKEN"),
#     model="openai/gpt-oss-20b",
#     api_type=OpenaiApiType.OpenAI,  
#     max_retries=20,
# )

# token_encoder = tiktoken.get_encoding("cl100k_base")

# INPUT_DIR = "/content/rag_exim/output/"   # path of output folder which has all parquete files
# COMMUNITY_REPORT_TABLE = "create_final_community_reports"
# ENTITY_TABLE = "create_final_nodes"
# ENTITY_EMBEDDING_TABLE = "create_final_entities"

# # community level in the Leiden community hierarchy from which we will load the community reports
# # higher value means we use reports from more fine-grained communities (at the cost of higher computation cost)
# COMMUNITY_LEVEL = 2

# # %%
# entity_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_TABLE}.parquet")
# report_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
# entity_embedding_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_EMBEDDING_TABLE}.parquet")

# reports = read_indexer_reports(report_df, entity_df, COMMUNITY_LEVEL)
# entities = read_indexer_entities(entity_df, entity_embedding_df, COMMUNITY_LEVEL)
# print(f"Report records: {len(report_df)}")
# report_df.head()

# # #### Build global context based on community reports
# context_builder = GlobalCommunityContext(
#     community_reports=reports,
#     entities=entities,  # default to None if you don't want to use community weights for ranking
#     token_encoder=token_encoder,
# )

# # #### Perform global search

# context_builder_params = {
#     "use_community_summary": False,  # False means using full community reports. True means using community short summaries.
#     "shuffle_data": True,
#     "include_community_rank": True,
#     "min_community_rank": 0,
#     "community_rank_name": "rank",
#     "include_community_weight": True,
#     "community_weight_name": "occurrence weight",
#     "normalize_community_weight": True,
#     "max_tokens": 3_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
#     "context_name": "Reports",
# }

# map_llm_params = {
#     "max_tokens": 1000,
#     "temperature": 0.0,
#     "response_format": {"type": "json_object"},
# }

# reduce_llm_params = {
#     "max_tokens": 2000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000-1500)
#     "temperature": 0.0,
# }torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.

# search_engine = GlobalSearch(
#     llm=llm,
#     context_builder=context_builder,
#     token_encoder=token_encoder,
#     max_data_tokens=12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
#     map_llm_params=map_llm_params,
#     reduce_llm_params=reduce_llm_params,
#     allow_general_knowledge=False,  # set this to True will add instruction to encourage the LLM to incorporate general knowledge in the response, which may increase hallucinations, but could be useful in some use cases.
#     json_mode=True,  # set this to False if your LLM model does not support JSON mode.
#     context_builder_params=context_builder_params,
#     concurrent_coroutines=10,
#     response_type="multiple-page report",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
# )

# query = "suggest me some blog about clude?"
# result = await search_engine.asearch(query)
# print(result.response)
# print("____________________________________")
# print(f"LLM calls: {result.llm_calls}. LLM tokens: {result.prompt_tokens}")